# graph-rag-multimodal: walkthrough

A thin notebook over the package. Every function lives in `src/graphrag`; nothing is defined here.
Run `graphrag ingest` and `graphrag index` first, and set `GROQ_API_KEY` (and `NEO4J_*` for the graph) in `.env`.

In [ ]:
from graphrag.config import Settings
from graphrag.retrieval.pipeline import Pipeline

settings = Settings()
pipeline = Pipeline.from_settings(settings, use_graph=settings.graph_enabled, use_llm=True)

In [ ]:
question = "What are the three types of user credibility on Twitter?"
result = pipeline.retrieve(question)
for hit in result.text_chunks:
    print(f"{hit.chunk_id:32} {hit.score:.4f} via {', '.join(hit.retrieved_by)}")

In [ ]:
answer = pipeline.answer_from_retrieval(question, result)
print(answer["answer"])
print("unsupported:", answer.get("unsupported_claims"))
print("confidence:", answer["confidence"], "| self-reported:", answer["confidence_self_reported"])

In [ ]:
# Fusion modes side by side on the same retrievers. The full original also had the
# BM25 tokenizer bug; reproduce it with `graphrag eval --preset original`.
from graphrag.retrieval.hybrid import HybridRetriever

for mode in ("legacy", "rrf"):
    r = HybridRetriever(pipeline.hybrid.dense, pipeline.hybrid.sparse, mode=mode)
    print(mode, [h.chunk_id for h in r.retrieve(question, top_k=5)])